#### 03 - Model Training & Selection 

**Goal: * Train and compare four models of increasing complexity, using 5-fold cross-validation and SMOTE to address class imbalamce. Select the best model based on validation AUC-ROC.

**Models (in order of complexity): *
1. Dummy Classifier - baseline floor 
2. Logistic Regression - interpretable referance point 
3. Random Forest - non-linear, gives feature importance
3. XGBoost - typically the strongest performer on tabular data

**Why this order matters: * each model must beat the one before it to justify its added complexity. If Logistic Regression doesn't beat Random Forest, the simpler model wins on the principle of parsimony.

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib

In [17]:
X_train = np.load('../data/processed/X_train.npy')
X_val   = np.load('../data/processed/X_val.npy')    
X_test  = np.load('../data/processed/X_test.npy')   
y_train = np.load('../data/processed/y_train.npy')
y_val   = np.load('../data/processed/y_val.npy')   
y_test  = np.load('../data/processed/y_test.npy') 

print(X_train.shape)
print(X_val.shape)  
print(X_test.shape) 
print(f'Churn rate - train: {y_train.mean():.3f}')

(4507, 43)
(1127, 43)
(1409, 43)
Churn rate - train: 0.265


#### Understanding SMOTE before using it


#### Understanding 5-fold cross-validation before using it

In [18]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#Scoring metrics
scoring = {'auc_roc'    : 'roc_auc',
           'precision'  : 'precision',
           'recall'     : 'recall',
           'f1'         : 'f1'}

#### Dummy Classifier 

In [19]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)

dummy_score = cross_validate(dummy, 
                             X_train, 
                             y_train, 
                             cv=cv_strategy, 
                             scoring=scoring, 
                             return_train_score=False)

for metric in scoring:
    print(f' {metric}   : {dummy_score[f'test_{metric}'].mean():.3f}')

 auc_roc   : 0.500
 precision   : 0.000
 recall   : 0.000
 f1   : 0.000


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3

#### Logistic Regression with SMOTE

In [25]:
logreg_pipeline = ImbPipeline(steps=[('smote', SMOTE(random_state=42)),
                                    ('model', LogisticRegression(max_iter=1000, random_state=42))])

logreg_scores = cross_validate(logreg_pipeline, 
                               X_train, 
                               y_train,
                               cv=cv_strategy,
                               scoring=scoring,
                               return_train_score=False)

for metric in scoring:
    print(f' {metric}  : {logreg_scores[f'test_{metric}'].mean():.3f}')

 auc_roc  : 0.845
 precision  : 0.524
 recall  : 0.799
 f1  : 0.633
